# Lab 21 · Track 3 — Fine-tuning LLM · Kaggle FULL run (unsloth · Qwen3.5-4B)

Chạy từ trên xuống, một notebook duy nhất, **không `git clone`** gì cả. Backbone:
`unsloth/Qwen3.5-4B`, train bằng unsloth (LoRA nhanh, mask assistant-only bằng
`train_on_responses_only`, tuỳ chọn 4-bit).

### Trước khi chạy
| Cần | Ở đâu |
|---|---|
| **GPU T4 x2** | Settings → Accelerator → GPU T4 x2 (notebook tự khoá về 1 card) |
| **Internet ON** | Settings → Internet → On |
| **Dataset** | Add Input → attach dataset chứa 5 file: `train_seed.jsonl`, `eval_target.jsonl`, `eval_regression.jsonl`, `holdout_secret.jsonl`, `checksums.json` |

### Nội dung
§1 môi trường · §2 config · §3 data + checksum · §4 chat template & mask (assistant-only,
bằng chứng) · §5 replay corpus (deck §14.3, đã tẩy trùng) · §6 baseline (a) naive prompt,
(b) prompt tối ưu — đo **trước khi train** · §7 train `correct` + `correct_replay` (unsloth
LoRA, all-linear, lr 10× full-FT, alpha=2r, batch<32) · §8 ba cấu hình **sai**
(`attn_only`, `wrong_lr`, `qlora`) cùng ngân sách step · §9 bốn nhóm điểm (target ·
regression · format · latency) + phán quyết hồi quy · §10 latency & cost ($/1k ticket,
break-even) · §11 chạy thử · §12 merge tuỳ chọn · §13 cổng kiểm tra + REPORT.md + zip.

## 1. Môi trường — khoá 1 GPU trước khi torch import, cài unsloth

In [ ]:
import os, pathlib, subprocess, sys

os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # PHẢI đặt trước khi import torch
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

WORK = pathlib.Path("/kaggle/working")
if not WORK.exists():
    WORK = pathlib.Path.cwd()
os.chdir(WORK)
for sub in ("data", "results", "adapters", "submission"):
    (WORK / sub).mkdir(exist_ok=True)
print("cwd:", WORK)

PKGS = [
    "unsloth", "unsloth_zoo",
    "transformers>=4.46", "trl>=0.12,<0.13", "peft>=0.13",
    "accelerate>=1.0", "datasets>=2.20", "bitsandbytes>=0.43",
]
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *PKGS],
                      capture_output=True, text=True)
print(proc.stdout[-1500:] or "pip: ok")
if proc.returncode:
    print(proc.stderr[-3000:])
    raise SystemExit("pip install thất bại. Kiểm tra Settings → Internet = On.")

import torch
print("torch", torch.__version__)
if not torch.cuda.is_available():
    raise SystemExit("Không thấy GPU. Settings → Accelerator → GPU T4 x2.")
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} · {p.total_memory/1024**3:.1f} GB · visible={torch.cuda.device_count()}")

## 2. CONFIG — mọi nút của lab

In [ ]:
MODEL_ID        = "unsloth/Qwen3.5-4B"
MAX_SEQ_LEN     = 1024          # ghi đè bằng p95 đo được ở §4.3
LOAD_IN_4BIT    = False         # tier "T4" mặc định 16bit; contrast `qlora` tự bật 4bit

EPOCHS          = 2             # CHUNG cho mọi run -> so được với nhau
PER_DEVICE_BATCH = 2
GRAD_ACCUM      = 8             # effective batch = 16 < 32 (deck §10.4)
LORA_R          = 16
LORA_ALPHA      = 32            # = 2r (deck §9.3)
LORA_LR         = 1e-4          # ~10x LR full-FT giả định 1e-5 (deck §10.3)
FULL_FT_LR_REF  = 1e-5
REPLAY_FRACTION = 0.05          # deck §14.3: 1-5%
SEED            = 42

RUN_REPLAY      = True
RUN_CONTRASTS   = True
RUN_MERGE       = False
FORCE_RETRAIN   = False
EVAL_LIMIT      = 0             # 0 = FULL bài nộp; vd 8 = tổng duyệt nhanh

EVAL_BATCH   = 4
MAX_NEW_TOK  = 160

# giá cho §10 -- GIẢ ĐỊNH, không phải số đo
GPU_HOURLY_USD   = 0.35
API_IN_USD_MTOK  = 0.15
API_OUT_USD_MTOK = 0.60

assert PER_DEVICE_BATCH * GRAD_ACCUM < 32, "batch hiệu dụng phải < 32 (deck §10.4)"
assert LORA_ALPHA == 2 * LORA_R, "alpha phải = 2r (deck §9.3)"
assert abs(LORA_LR / FULL_FT_LR_REF - 10.0) < 1e-9, "LoRA LR không ở thang 10x (deck §10.3)"
SMOKE = bool(EVAL_LIMIT) or LOAD_IN_4BIT
print(f"model={MODEL_ID}  effective_batch={PER_DEVICE_BATCH*GRAD_ACCUM}  "
      f"lr={LORA_LR}  r={LORA_R}  alpha={LORA_ALPHA}  replay={REPLAY_FRACTION:.0%}  "
      f"eval_limit={EVAL_LIMIT or 'FULL'}")

## 3. Dữ liệu — tìm dataset đã attach và kiểm checksum (CRLF-tolerant)

Checksum được tính trên nội dung đã chuẩn hoá **LF**; một checkout Windows cho ra
CRLF — cùng dữ liệu, khác byte, khác SHA thô. Ô này băm cả hai dạng, chỉ FAIL khi
**cả hai** đều lệch, để không tố oan một dataset upload từ Windows.

In [ ]:
import hashlib, json, shutil

NEEDED = ["train_seed.jsonl", "eval_target.jsonl", "eval_regression.jsonl",
          "holdout_secret.jsonl"]
DATA = WORK / "data"

def find_dataset_root():
    roots = list(pathlib.Path("/kaggle/input").glob("*")) \
            if pathlib.Path("/kaggle/input").exists() else []
    cands = []
    for root in roots:
        for d in [root, *(p for p in root.rglob("*") if p.is_dir())]:
            if (d / "train_seed.jsonl").exists():
                cands.append(d)
    if not cands:
        if all((DATA / f).exists() for f in NEEDED):
            return DATA
        raise SystemExit("Không tìm thấy train_seed.jsonl trong /kaggle/input. "
                          "Add Input -> attach dataset rồi chạy lại ô này.")
    cands.sort(key=lambda d: sum((d / f).exists() for f in NEEDED), reverse=True)
    return cands[0]

SRC_DATA = find_dataset_root()
print("dataset:", SRC_DATA)
for f in NEEDED + ["checksums.json"]:
    s = SRC_DATA / f
    if s.exists():
        if s.resolve() != (DATA / f).resolve():
            shutil.copy2(s, DATA / f)
    elif f in NEEDED:
        raise SystemExit(f"thiếu {f} trong dataset đã attach")

def sha16(path):
    b = path.read_bytes()
    return (hashlib.sha256(b).hexdigest()[:16],
            hashlib.sha256(b.replace(b"\r\n", b"\n")).hexdigest()[:16])

ref = json.loads((DATA / "checksums.json").read_text(encoding="utf-8")) \
      if (DATA / "checksums.json").exists() else {}
integrity = {"dataset_dir": str(SRC_DATA), "files": {}, "drift": []}
for f in NEEDED:
    p = DATA / f
    raw, lf = sha16(p)
    n = sum(1 for line in p.open(encoding="utf-8") if line.strip())
    want = ref.get(f)
    ok = want is None or want in (raw, lf)
    integrity["files"][f] = {"rows": n, "sha_raw": raw, "sha_lf": lf,
                             "expected": want, "ok": ok}
    if not ok:
        integrity["drift"].append(f)
    print(f"  {f:<24} {n:>4} dòng  sha_raw={raw}  sha_lf={lf}  expected={want}  "
          f"[{'ok' if ok else 'DRIFT'}]")
if integrity["drift"]:
    raise SystemExit(f"checksum lệch: {integrity['drift']}. Attach lại dataset gốc.")
print("\ncorpus khớp checksum.")

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

train_raw = load_jsonl(DATA / "train_seed.jsonl")
target_all = load_jsonl(DATA / "eval_target.jsonl")
regression_all = load_jsonl(DATA / "eval_regression.jsonl")
holdout_all = load_jsonl(DATA / "holdout_secret.jsonl")
L = EVAL_LIMIT or None
target = target_all[:L] if L else target_all
regression = regression_all[:L] if L else regression_all
holdout = holdout_all[:L] if L else holdout_all
print(f"train={len(train_raw)}  target={len(target)}/{len(target_all)}  "
      f"regression={len(regression)}/{len(regression_all)}  "
      f"holdout={len(holdout)}/{len(holdout_all)}")

## 4. Model (unsloth) + chat template + **mask loss có bằng chứng**

`FastLanguageModel.from_pretrained` nạp Qwen3.5-4B qua unsloth (nhanh hơn, VRAM thấp
hơn HF thuần). Mask assistant-only dùng `train_on_responses_only` của unsloth: nó cắt
theo **chuỗi ranh giới template** (không phải trừ token), nên tự đúng với mọi chat
template — nhưng vẫn phải **chứng minh bằng số** rằng câu hỏi bị che và câu trả lời
thì không, trước khi tiêu một phút GPU nào cho train.

In [ ]:
!pip uninstall -y unsloth unsloth_zoo transformers trl peft accelerate bitsandbytes xformers tokenizers 2>/dev/null

!pip install --no-deps --upgrade \
    "unsloth[kaggle-new]" \
    "unsloth_zoo"

!pip install --no-deps --upgrade \
    "transformers==4.55.0" \
    "tokenizers>=0.21,<0.22" \
    "trl==0.19.0" \
    "peft==0.15.2" \
    "accelerate==1.7.0" \
    "bitsandbytes==0.45.5"

!pip install --no-deps xformers

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

In [ ]:
import unsloth  # MUST be first, before torch/transformers/trl/peft
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import torch

MODEL_ID = "unsloth/Qwen3-8B-bnb-4bit"  # example — set to your actual model id
LOAD_IN_4BIT = True
MAX_SEQ_LEN = 2048

def load_model(model_id=MODEL_ID, load_in_4bit=LOAD_IN_4BIT, max_seq_length=MAX_SEQ_LEN):
    model, tok = FastLanguageModel.from_pretrained(
        model_name=model_id,
        max_seq_length=max_seq_length,
        load_in_4bit=load_in_4bit,
        dtype=None,
    )
    tok = get_chat_template(tok, chat_template="qwen3")
    return model, tok

base, tok = load_model()
print("loaded:", MODEL_ID, "| eos:", tok.eos_token, "| 4bit:", LOAD_IN_4BIT)

In [ ]:
NAIVE_PROMPT = "Phân loại ticket sau."
OPTIMIZED_PROMPT = (
    "Phân loại ticket chăm sóc khách hàng sau thành JSON với đúng 4 khóa: "
    "intent, urgency, product, sentiment. Chỉ trả về JSON, không giải thích.\n\n"
    "intent thuộc: doi_tra | van_chuyen | hoan_tien | san_pham_loi | hoi_thong_tin\n"
    "urgency thuộc: cao | trung_binh | thap\n"
    "sentiment thuộc: tieu_cuc | trung_tinh | tich_cuc\n"
    "product: tên sản phẩm xuất hiện trong ticket."
)

def to_messages(row, system=None):
    sys_prompt = system if system is not None else row.get("instruction", OPTIMIZED_PROMPT)
    msgs = [{"role": "system", "content": sys_prompt},
            {"role": "user", "content": row["input"]}]
    if "output" in row:
        msgs.append({"role": "assistant", "content": row["output"]})
    return msgs

rendered = tok.apply_chat_template(to_messages(train_raw[0]), tokenize=False,
                                   add_generation_prompt=False)
print("--- ví dụ chat template đã render ---")
print(rendered[:600])
think_ok = "<think>" not in rendered.split(train_raw[0]["output"][:20])[0] or True
template_check = {"model": MODEL_ID, "rendered_preview": rendered[:800]}
(WORK / "results").mkdir(exist_ok=True)
json.dump(template_check, open(WORK / "results" / "template_check.json", "w",
                               encoding="utf-8"), ensure_ascii=False, indent=2)

### 4.1 Bằng chứng mask: dựng một batch train, so chuỗi supervise vs chuỗi bị che

In [ ]:
# ============================================================
# CELL 1: Import (chỉ cần chạy nếu chưa import trong session này)
# ============================================================
import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import torch
import json
import random
from pathlib import Path

In [ ]:
# ============================================================
# CELL 2: Kiểm tra biến nào đã có / đang thiếu trong session hiện tại
# ============================================================
for name in ["base", "tok", "train_raw", "WORK", "LORA_LR", "SEED",
             "PER_DEVICE_BATCH", "GRAD_ACCUM", "MAX_SEQ_LEN",
             "to_messages", "OPTIMIZED_PROMPT"]:
    print(name, "->", "OK" if name in dir() else "❌ THIẾU - cần chạy lại cell tương ứng")

In [ ]:
!pip install --no-deps --upgrade "peft>=0.16.0"

In [ ]:
import importlib
import peft
importlib.reload(peft)
from peft import LoraConfig
import inspect
print("target_parameters" in inspect.signature(LoraConfig.__init__).parameters)

In [ ]:
# ============================================================
# CELL 3: Gắn LoRA adapter (bỏ qua nếu base đã có peft_config)
# ============================================================
if not hasattr(base, "peft_config"):
    base = FastLanguageModel.get_peft_model(
        base,
        r=16,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

print(type(base))
print("has peft_config:", hasattr(base, "peft_config"))

In [2]:
# ============================================================
# CELL 4: Định nghĩa to_hf_dataset và make_trainer
# ============================================================
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

def to_hf_dataset(rows, tok, system_fn=None):
    texts = []
    for r in rows:
        sys_p = system_fn(r) if system_fn else r.get("instruction", OPTIMIZED_PROMPT)
        texts.append(tok.apply_chat_template(
            to_messages(r, system=sys_p),
            tokenize=False, add_generation_prompt=False
        ))
    return Dataset.from_dict({"text": texts})


def make_trainer(model, tok, ds, out_dir, *, max_steps, lr, seed=SEED, val_ds=None):
    args = SFTConfig(
        per_device_train_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=max(1, max_steps // 10),
        max_steps=max_steps,
        learning_rate=lr,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=seed,
        output_dir=str(out_dir),
        report_to="none",
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        packing=False,
        eval_strategy="steps" if val_ds is not None else "no",
        eval_steps=max(1, max_steps // 6) if val_ds is not None else None,
    )
    return SFTTrainer(
        model=model, tokenizer=tok, train_dataset=ds,
        eval_dataset=val_ds, args=args,
    )

print("✅ to_hf_dataset, make_trainer đã sẵn sàng")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


NameError: name 'SEED' is not defined

In [ ]:
# ============================================================
# Probe run
# ============================================================
_probe_ds = to_hf_dataset(train_raw[:2], tok)
_probe_trainer = make_trainer(
    base, tok, _probe_ds, WORK / "results" / "_probe",
    max_steps=1, lr=LORA_LR,
)
_probe_trainer = train_on_responses_only(
    _probe_trainer,
    instruction_part=_instruction_part,
    response_part=_response_part,
)

# Dùng dataloader KHÔNG shuffle để batch[0] chắc chắn khớp train_raw[0]
from torch.utils.data import DataLoader

_collator = _probe_trainer.data_collator
_train_ds_processed = _probe_trainer.train_dataset
_probe_loader = DataLoader(
    _train_ds_processed,
    batch_size=_probe_trainer.args.per_device_train_batch_size,
    shuffle=False,          # <-- điểm mấu chốt của fix
    collate_fn=_collator,
)

batch = next(iter(_probe_loader))
ids, labels = batch["input_ids"][0], batch["labels"][0]
supervised_ids = ids[labels != -100]
masked_ids = ids[labels == -100]
supervised_text = tok.decode(supervised_ids, skip_special_tokens=False)
masked_text = tok.decode(masked_ids, skip_special_tokens=False)

answer_frag = train_raw[0]["output"][:30]
question_frag = train_raw[0]["input"][:30]

mask_proof = {
    "instruction_part": _instruction_part,
    "response_part": _response_part,
    "n_supervised": int((labels != -100).sum()),
    "n_total": int(labels.numel()),
    "supervised_fraction": round(float((labels != -100).float().mean()), 4),
    "answer_is_supervised": answer_frag in supervised_text,
    "question_is_masked": question_frag not in supervised_text,
    "supervised_preview": supervised_text[:300],
    "masked_preview": masked_text[:200],
}

print(json.dumps(
    {k: v for k, v in mask_proof.items() if not k.endswith("preview")},
    ensure_ascii=False, indent=2
))
print("\n--- SUPERVISED PREVIEW ---")
print(mask_proof["supervised_preview"])
print("\n--- MASKED PREVIEW ---")
print(mask_proof["masked_preview"])

if not mask_proof["answer_is_supervised"]:
    print("\n⚠️ answer_frag không khớp — so sánh trực tiếp:")
    print("answer_frag:", repr(answer_frag))
    print("supervised_text[:200]:", repr(supervised_text[:200]))

assert mask_proof["answer_is_supervised"], "câu trả lời KHÔNG nằm trong loss — mask sai"
assert mask_proof["question_is_masked"], "câu hỏi ĐANG nằm trong loss — mask sai"
assert mask_proof["supervised_fraction"] < 0.95, "gần như mọi token vào loss — đang train cả prompt"

json.dump(
    mask_proof,
    open(WORK / "results" / "mask_proof.json", "w", encoding="utf-8"),
    ensure_ascii=False, indent=2,
)

del _probe_trainer, _probe_ds, _probe_loader

### 4.2 `max_length` từ p95 đo được, và split train/val seed 42

In [ ]:
import random

lengths = [len(tok(tok.apply_chat_template(to_messages(r), tokenize=False))["input_ids"])
           for r in train_raw]
lengths.sort()
p95 = lengths[int(0.95 * (len(lengths) - 1))]
token_stats = {"n": len(lengths), "min": lengths[0], "p50": lengths[len(lengths)//2],
               "p95": p95, "max": lengths[-1], "suggested_max_length": int(p95 * 1.15)}
print(json.dumps(token_stats, ensure_ascii=False, indent=2))
json.dump(token_stats, open(WORK / "results" / "token_stats.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
if token_stats["suggested_max_length"] > MAX_SEQ_LEN:
    print(f"⚠ p95 gợi ý max_length={token_stats['suggested_max_length']} > MAX_SEQ_LEN hiện "
          f"tại ({MAX_SEQ_LEN}). Cân nhắc tăng MAX_SEQ_LEN ở §2 rồi chạy lại từ §4.")

rng = random.Random(SEED)
shuffled = train_raw[:]
rng.shuffle(shuffled)
n_val = max(1, int(0.1 * len(shuffled)))
val_rows, train_rows = shuffled[:n_val], shuffled[n_val:]
print(f"train={len(train_rows)}  val={len(val_rows)}  (seed={SEED})")

## 5. Replay corpus (deck §14.3) — trộn dữ liệu tổng quát, đã **tẩy trùng** với eval

Repo gốc đo được: fine-tune chỉ trên ticket khiến general capability rơi mạnh vì mọi
input đều là ticket và mọi output đều là JSON. Cách chữa (deck §14.3) là trộn lại 1–5%
dữ liệu tổng quát. Rủi ro là **nhiễm tập eval**: nếu dữ liệu trộn trùng câu hỏi trong
`eval_regression.jsonl`, điểm regression tăng vì học thuộc đề chứ không phải vì giữ
được năng lực. Pool dưới đây được kiểm bằng (1) không trùng chính xác sau chuẩn hoá,
(2) Jaccard từ vựng ≤ 0.6 với *mọi* câu trong `eval_regression.jsonl`.

In [ ]:
REPLAY_POOL = [
    {"input": "Việt Nam có bao nhiêu tỉnh thành hiện nay?",
     "output": "Việt Nam hiện có 63 tỉnh, thành phố trực thuộc trung ương."},
    {"input": "Ai là tác giả của Truyện Kiều?",
     "output": "Truyện Kiều là tác phẩm của đại thi hào Nguyễn Du."},
    {"input": "Nước nào có diện tích lớn nhất thế giới?",
     "output": "Nga là quốc gia có diện tích lớn nhất thế giới."},
    {"input": "Giải thích ngắn gọn hiện tượng nhật thực.",
     "output": "Nhật thực xảy ra khi Mặt Trăng đi qua giữa Trái Đất và Mặt Trời, che khuất ánh sáng mặt trời."},
    {"input": "Viết một câu chúc Tết ngắn gọn.",
     "output": "Chúc mừng năm mới, an khang thịnh vượng, vạn sự như ý!"},
    {"input": "Công thức tính diện tích hình tròn là gì?",
     "output": "Diện tích hình tròn = pi nhân bán kính bình phương (S = πr²)."},
    {"input": "Kể tên ba loại trái cây nhiệt đới phổ biến ở Việt Nam.",
     "output": "Xoài, chôm chôm và sầu riêng là ba loại trái cây nhiệt đới phổ biến."},
    {"input": "Ngọn núi cao nhất Việt Nam tên là gì?",
     "output": "Fansipan, thuộc dãy Hoàng Liên Sơn, là ngọn núi cao nhất Việt Nam."},
    {"input": "1 giờ có bao nhiêu phút?",
     "output": "1 giờ có 60 phút."},
    {"input": "Nêu tên một loại hình nghệ thuật truyền thống của Việt Nam.",
     "output": "Ca trù là một loại hình nghệ thuật truyền thống lâu đời của Việt Nam."},
    {"input": "Ai được xem là người phát minh ra bóng đèn sợi đốt?",
     "output": "Thomas Edison thường được ghi nhận là người phát minh ra bóng đèn sợi đốt thực dụng."},
    {"input": "Tóm tắt ngắn gọn vòng đời của con bướm.",
     "output": "Bướm trải qua bốn giai đoạn: trứng, sâu, nhộng, và bướm trưởng thành."},
    {"input": "Đơn vị đo nhiệt độ phổ biến ở Việt Nam là gì?",
     "output": "Độ C (Celsius) là đơn vị đo nhiệt độ phổ biến ở Việt Nam."},
    {"input": "Cho biết tên một hồ nước ngọt lớn ở Việt Nam.",
     "output": "Hồ Ba Bể ở Bắc Kạn là một hồ nước ngọt tự nhiên lớn ở Việt Nam."},
    {"input": "Giải thích ngắn vì sao lá cây có màu xanh.",
     "output": "Lá cây có màu xanh vì chứa diệp lục, chất hấp thụ ánh sáng đỏ và xanh dương, phản xạ ánh sáng xanh lục."},
]
print(f"replay pool: {len(REPLAY_POOL)} mẫu (nguồn tổng quát, ngoài miền triage)")

In [ ]:
import unicodedata, re

def _norm(s):
    s = unicodedata.normalize("NFC", s.strip().lower())
    return re.sub(r"\s+", " ", s)

def _jaccard(a, b):
    sa, sb = set(_norm(a).split()), set(_norm(b).split())
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

JACCARD_THRESHOLD = 0.6

# === Lọc bỏ các item trong REPLAY_POOL bị trùng (exact hoặc quá giống)
# với regression_all TRƯỚC khi assert, thay vì chỉ phát hiện rồi fail cứng ===
regression_norms = {_norm(ev["instruction"]) for ev in regression_all}

clean_pool = []
removed_exact = []
removed_near = []

for cand in REPLAY_POOL:
    cand_norm = _norm(cand["input"])

    if cand_norm in regression_norms:
        removed_exact.append(cand["input"])
        continue

    max_j = 0.0
    max_j_match = None
    for ev in regression_all:
        j = _jaccard(cand["input"], ev["instruction"])
        if j > max_j:
            max_j, max_j_match = j, ev["instruction"]

    if max_j > JACCARD_THRESHOLD:
        removed_near.append((cand["input"], max_j_match, round(max_j, 3)))
        continue

    clean_pool.append(cand)

print(f"REPLAY_POOL gốc: {len(REPLAY_POOL)}")
print(f"Loại vì trùng chính xác: {len(removed_exact)}")
print(f"Loại vì quá giống (jaccard > {JACCARD_THRESHOLD}): {len(removed_near)}")
print(f"REPLAY_POOL sạch: {len(clean_pool)}")

if removed_exact:
    print("\n--- Ví dụ item bị loại (trùng chính xác) ---")
    for x in removed_exact[:5]:
        print(" -", x[:80])

if removed_near:
    print("\n--- Ví dụ item bị loại (quá giống) ---")
    for cand, match, j in removed_near[:5]:
        print(f" - jaccard={j} | cand={cand[:60]!r} | match={match[:60]!r}")

REPLAY_POOL = clean_pool  # ghi đè bằng pool đã khử trùng lặp

# === Chạy lại kiểm tra deconflict trên pool ĐÃ SẠCH, giờ phải pass ===
exact_collisions = 0
worst = (0.0, None, None)
for cand in REPLAY_POOL:
    for ev in regression_all:
        if _norm(cand["input"]) == _norm(ev["instruction"]):
            exact_collisions += 1
        j = _jaccard(cand["input"], ev["instruction"])
        if j > worst[0]:
            worst = (j, cand["input"], ev["instruction"])

decon = {
    "exact_collisions": exact_collisions,
    "worst_jaccard": round(worst[0], 3),
    "jaccard_threshold": JACCARD_THRESHOLD,
    "worst_pair": (worst[1], worst[2]),
    "n_removed_exact": len(removed_exact),
    "n_removed_near": len(removed_near),
    "pool_size_after_dedup": len(REPLAY_POOL),
}

assert exact_collisions == 0, "replay pool vẫn trùng chính xác với câu hỏi regression sau khi lọc — kiểm tra lại logic lọc"
assert worst[0] <= JACCARD_THRESHOLD, f"replay pool vẫn quá giống câu regression sau khi lọc (jaccard={worst[0]:.2f})"

print("\n" + json.dumps(decon, ensure_ascii=False, indent=2))

json.dump(decon, open(WORK / "results" / "replay_manifest.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)


def mix_replay(rows, pool, fraction, seed=SEED):
    rng = random.Random(seed)
    n_replay = max(1, round(len(rows) * fraction / (1 - fraction))) if fraction > 0 else 0
    picked = [pool[i % len(pool)] for i in rng.sample(range(len(pool) * 10), n_replay)] \
             if n_replay else []
    replay_rows = [{"instruction": OPTIMIZED_PROMPT, "input": r["input"],
                    "output": r["output"], "label": None} for r in picked]
    mixed = rows + replay_rows
    rng.shuffle(mixed)
    return mixed, len(replay_rows)


mixed_rows, n_replay_added = mix_replay(train_rows, REPLAY_POOL, REPLAY_FRACTION)
print(f"train={len(train_rows)}  +replay={n_replay_added}  -> mixed={len(mixed_rows)} "
      f"({n_replay_added/len(mixed_rows):.1%} thực tế)")

## 6. Sinh có đo latency, và **bốn nhóm điểm**

`generate_measured` đo latency thật (warm-up bị loại khỏi trung bình), đếm token
prompt/output — đây là số nguyên liệu cho phần cost ở §10.

In [ ]:
import time

def generate_measured(model, tok, inputs, system, *, max_new_tokens=MAX_NEW_TOK,
                      batch_size=EVAL_BATCH, warmup=True, label=""):
    FastLanguageModel.for_inference(model)
    prompts = [tok.apply_chat_template(to_messages({"input": x}, system=system),
                                       tokenize=False, add_generation_prompt=True)
               for x in inputs]
    outs, lat_ms, ptoks, ntoks = [], [], [], []
    for bi in range(0, len(prompts), batch_size):
        batch = prompts[bi:bi + batch_size]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True,
                  max_length=MAX_SEQ_LEN).to(model.device)
        is_warmup = warmup and bi == 0
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tok.pad_token_id or tok.eos_token_id)
        torch.cuda.synchronize()
        dt = (time.perf_counter() - t0) * 1000
        new_tokens = gen[:, enc["input_ids"].shape[1]:]
        texts = tok.batch_decode(new_tokens, skip_special_tokens=True)
        outs.extend(texts)
        if not is_warmup:
            lat_ms.append(dt / len(batch))
            ptoks.append(enc["input_ids"].shape[1])
            ntoks.append(new_tokens.shape[1])
    stats = {
        "mean_latency_ms": round(sum(lat_ms) / max(1, len(lat_ms)), 1),
        "prompt_tokens_mean": round(sum(ptoks) / max(1, len(ptoks)), 1),
        "new_tokens_mean": round(sum(ntoks) / max(1, len(ntoks)), 1),
        "n": len(inputs), "label": label,
    }
    return outs, stats

In [ ]:
TRIAGE_KEYS = ["intent", "urgency", "product", "sentiment"]

def parse_json_obj(text):
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def field_accuracy(preds, rows):
    acc = {k: [] for k in TRIAGE_KEYS}
    unparsed = 0
    for p, r in zip(preds, rows):
        obj = parse_json_obj(p)
        if obj is None:
            unparsed += 1
            continue
        for k in TRIAGE_KEYS:
            acc[k].append(1.0 if str(obj.get(k, "")).strip() == str(r["label"].get(k, "")).strip()
                          else 0.0)
    out = {k: (sum(v) / len(v) if v else None) for k, v in acc.items()}
    out["unparsed"] = unparsed
    return out

def triage_field_accuracy(pred, label):
    obj = parse_json_obj(pred)
    if obj is None:
        return 0.0
    hits = sum(1 for k in TRIAGE_KEYS if str(obj.get(k, "")).strip() == str(label.get(k, "")).strip())
    return hits / len(TRIAGE_KEYS)

def format_valid_rate(preds):
    ok = 0
    for p in preds:
        obj = parse_json_obj(p)
        if obj is not None and all(k in obj for k in TRIAGE_KEYS):
            ok += 1
    return ok / max(1, len(preds))

def keyword_recall(preds, rows):
    scores = []
    for p, r in zip(preds, rows):
        kws = r.get("keywords", [])
        if not kws:
            continue
        hit = sum(1 for kw in kws if _norm(kw) in _norm(p))
        scores.append(hit / len(kws))
    return sum(scores) / len(scores) if scores else 0.0

class Score:
    def __init__(self, target, regression, fmt, latency_ms, n, extra):
        self.target, self.regression, self.format = target, regression, fmt
        self.latency_ms, self.n, self.extra = latency_ms, n, extra
    def as_dict(self):
        return {"target": round(self.target, 4), "regression": round(self.regression, 4),
                "format": round(self.format, 4), "latency_ms": round(self.latency_ms, 1),
                "n": self.n}

def score_version(target_rows, target_preds, regression_rows, regression_preds,
                  latency_ms, extra=None):
    t = sum(triage_field_accuracy(p, r["label"]) for p, r in
            zip(target_preds, target_rows)) / max(1, len(target_rows))
    reg = keyword_recall(regression_preds, regression_rows) if regression_rows else 0.0
    fmt = format_valid_rate(target_preds)
    e = dict(extra or {})
    e["fields"] = field_accuracy(target_preds, target_rows)
    return Score(t, reg, fmt, latency_ms, len(target_rows), e)

REGRESSION_TOLERANCE = 0.02

def regression_gate(candidate, baseline):
    reasons = []
    beats_target = candidate.target > baseline.target
    reasons.append(f"target: {candidate.target:.3f} vs (b) {baseline.target:.3f} "
                   f"-> {'thắng' if beats_target else 'KHÔNG thắng'}")
    keeps_regression = (baseline.regression - candidate.regression) <= REGRESSION_TOLERANCE
    reasons.append(f"regression: {candidate.regression:.3f} vs (b) {baseline.regression:.3f} "
                   f"(dung sai {REGRESSION_TOLERANCE}) -> "
                   f"{'giữ được' if keeps_regression else 'TỤT quá ngưỡng'}")
    class Verdict:
        passed = beats_target and keeps_regression
    v = Verdict()
    v.reasons = reasons
    return v

def comparison_table(scores: dict):
    return [{"version": k, **v.as_dict()} for k, v in scores.items()]

def md_table(rows, cols=None):
    if not rows:
        return "_(trống)_"
    cols = cols or list(rows[0].keys())
    head = "| " + " | ".join(cols) + " |"
    sep = "| " + " | ".join("---" for _ in cols) + " |"
    body = "\n".join("| " + " | ".join(str(r.get(c, "")) for c in cols) + " |" for r in rows)
    return "\n".join([head, sep, body])

### 6.1 Baselines (a) naive prompt và (b) prompt tối ưu — đo và **đóng băng** trước khi train

In [ ]:
SCORES, PREDS, STATS, HOLD = {}, {}, {}, {}
V_A, V_B, V_C, V_CR = "(a) naive prompt", "(b) optimized prompt", "(c) fine-tune", "(c+) fine-tune+replay"

def eval_pass(model, tok, version, system, *, full=True):
    tp, st = generate_measured(model, tok, [r["input"] for r in target], system,
                               label=f"{version}/target")
    rp = hp = sp = gp = []
    if full:
        rp, _ = generate_measured(model, tok, [r["instruction"] for r in regression], None,
                                  max_new_tokens=96, warmup=False, label=f"{version}/regression")
        hp, _ = generate_measured(model, tok, [r["input"] for r in holdout], system,
                                  warmup=False, label=f"{version}/holdout")
    sc = score_version(target, tp, regression if full else [], rp, st["mean_latency_ms"],
                       extra={"generation": st, "system_prompt": (system or "")[:60],
                              "scored_regression": full})
    SCORES[version] = sc
    STATS[version] = st
    PREDS[version] = {"target": tp, "regression": rp, "holdout": hp}
    if hp:
        HOLD[version] = {"target": round(sum(triage_field_accuracy(p, r["label"])
                                              for p, r in zip(hp, holdout)) / len(holdout), 4),
                         "n": len(holdout)}
    line = (f"{version:<26} target={sc.target:.3f}  format={sc.format:.3f}  "
            f"{sc.latency_ms:7.0f} ms/mẫu")
    if full:
        line += f"  regression={sc.regression:.3f}"
        if hp:
            line += f"  holdout={HOLD[version]['target']:.3f}"
    print(line)
    return sc

eval_pass(base, tok, V_A, NAIVE_PROMPT)
eval_pass(base, tok, V_B, OPTIMIZED_PROMPT)

d = SCORES[V_B].target - SCORES[V_A].target
print(f"\nprompt tối ưu đổi được {d:+.3f} target — đây là mốc (b) mà fine-tune phải vượt.")
if d <= 0:
    print("⚠ (b) KHÔNG hơn (a). Sửa OPTIMIZED_PROMPT trước khi train.")

frozen = {"baseline_a": SCORES[V_A].as_dict(), "baseline_b": SCORES[V_B].as_dict(),
          "n_target": len(target), "n_regression": len(regression), "n_holdout": len(holdout),
          "smoke_mode": SMOKE}
json.dump(frozen, open(WORK / "results" / "baselines_frozen.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)

import gc
del base
gc.collect(); torch.cuda.empty_cache()

## 7. Huấn luyện — unsloth LoRA, **một hàm cho mọi run**

Tất cả các run (`correct`, `correct_replay`, và 3 contrast ở §8) đi qua cùng
`train_one()` và **cùng ngân sách step**, chỉ đổi đúng một biến mỗi lần. `eval_dataset`
dùng `val_rows` từ §4.2 để loss held-out là số đo, không phải suy đoán.

In [ ]:
ADAPTERS = WORK / "adapters"

def planned_steps(n_rows, epochs=EPOCHS):
    eff = PER_DEVICE_BATCH * GRAD_ACCUM
    return max(1, (n_rows * epochs) // eff)

STEPS = planned_steps(len(train_rows))
print(f"{len(train_rows)} mẫu  ·  effective batch {PER_DEVICE_BATCH*GRAD_ACCUM}  "
      f"·  epochs {EPOCHS}  ->  {STEPS} optimizer step (MỌI run dùng đúng con số này)")

def resolve_target_modules(model, placement):
    if placement == "attn_only":
        return ["q_proj", "v_proj"]
    return ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

def train_one(key, rows, steps, *, target_modules, r, alpha, lr, load_in_4bit=False,
             val_records=None):
    adir = ADAPTERS / key
    if (adir / "adapter_model.safetensors").exists() and not FORCE_RETRAIN:
        print(f"bỏ qua {key}: đã có {adir}")
        return None

    print("=" * 70, f"\nRUN {key}: modules={target_modules} r={r} alpha={alpha} "
          f"lr={lr} 4bit={load_in_4bit}")
    model, tk = load_model(load_in_4bit=load_in_4bit)
    model = FastLanguageModel.get_peft_model(
        model, r=r, lora_alpha=alpha, lora_dropout=0.0, bias="none",
        target_modules=target_modules, use_gradient_checkpointing="unsloth",
        random_state=SEED)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  trainable ≈ {trainable/1e6:.2f} M")

    ds = to_hf_dataset(rows, tk)
    val_ds = to_hf_dataset(val_records, tk) if val_records else None
    trainer = make_trainer(model, tk, ds, adir, max_steps=steps, lr=lr, val_ds=val_ds)
    trainer = train_on_responses_only(
        trainer, instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n")

    t0 = time.perf_counter()
    res = trainer.train()
    elapsed = time.perf_counter() - t0

    model.save_pretrained(adir)
    tk.save_pretrained(adir)
    print(f"  saved -> {adir}  ({elapsed:.0f}s)")

    hist = trainer.state.log_history
    train_losses = [h["loss"] for h in hist if "loss" in h]
    eval_losses = [h["eval_loss"] for h in hist if "eval_loss" in h]
    row = {"run": key, "r": r, "lora_alpha": alpha, "learning_rate": lr,
           "load_in_4bit": load_in_4bit, "trainable_params": trainable,
           "max_steps": steps, "n_train_examples": len(ds),
           "final_train_loss": round(train_losses[-1], 4) if train_losses else None,
           "final_eval_loss": round(eval_losses[-1], 4) if eval_losses else None,
           "train_seconds": round(elapsed, 1),
           "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2)}
    RUNS_ROWS.append(row)
    json.dump(row, open(WORK / "results" / f"run_{key}.json", "w", encoding="utf-8"),
              ensure_ascii=False, indent=2)
    print(json.dumps(row, ensure_ascii=False, indent=2))

    del trainer, model
    gc.collect(); torch.cuda.empty_cache()
    return row

RUNS_ROWS = []

In [ ]:
row_correct = train_one("correct", train_rows, STEPS,
                        target_modules=resolve_target_modules(None, "all"),
                        r=LORA_R, alpha=LORA_ALPHA, lr=LORA_LR, val_records=val_rows)

### 7b. `correct_replay` — cùng LoRA, khác **dữ liệu** (deck §14.3)

In [ ]:
row_replay = None
if RUN_REPLAY:
    row_replay = train_one("correct_replay", mixed_rows, STEPS,
                           target_modules=resolve_target_modules(None, "all"),
                           r=LORA_R, alpha=LORA_ALPHA, lr=LORA_LR, val_records=val_rows)
else:
    print("RUN_REPLAY=False — bỏ §7b.")

## 8. Ba cấu hình **sai** — cùng số step, mỗi lần đổi đúng một biến

| Run | Đổi gì | Kỳ vọng |
|---|---|---|
| `attn_only` | chỉ q,v — rank nâng lên để **khớp số tham số** với `correct` | thua `correct` |
| `wrong_lr` | LR thang full-FT (÷10) | loss gần như phẳng |
| `qlora` | 4-bit thay 16-bit | nhẹ hơn, chất lượng thấp hơn chút |

Đừng xếp hạng bằng `final_train_loss` — §9.3 chấm cả ba trên tập target, thang đo thật.

In [ ]:
def matched_rank(base_r, base_modules, target_modules):
    # xấp xỉ: tỉ lệ nghịch với số module bị thu hẹp, làm tròn tới bội của 4
    ratio = len(base_modules) / max(1, len(target_modules))
    return max(4, int(round(base_r * ratio / 4)) * 4)

if RUN_CONTRASTS:
    attn_r = matched_rank(LORA_R, resolve_target_modules(None, "all"),
                          resolve_target_modules(None, "attn_only"))
    train_one("attn_only", train_rows, STEPS,
              target_modules=resolve_target_modules(None, "attn_only"),
              r=attn_r, alpha=2 * attn_r, lr=LORA_LR, val_records=val_rows)
    train_one("wrong_lr", train_rows, STEPS,
              target_modules=resolve_target_modules(None, "all"),
              r=LORA_R, alpha=LORA_ALPHA, lr=LORA_LR / 10, val_records=val_rows)
    train_one("qlora", train_rows, STEPS,
              target_modules=resolve_target_modules(None, "all"),
              r=LORA_R, alpha=LORA_ALPHA, lr=LORA_LR, load_in_4bit=True,
              val_records=val_rows)
else:
    print("RUN_CONTRASTS=False — bỏ §8.")

seen = {r["run"]: r for r in RUNS_ROWS}
GRADED_KEYS = ["correct", "correct_replay", "attn_only", "wrong_lr", "qlora"]
runs_rows = [seen[k] for k in GRADED_KEYS if k in seen]
print()
print(md_table(runs_rows, ["run", "r", "lora_alpha", "learning_rate", "load_in_4bit",
                           "trainable_params", "max_steps", "final_train_loss",
                           "final_eval_loss", "train_seconds", "peak_vram_gb"]))
budgets = {r["run"]: r["max_steps"] for r in runs_rows}
print(f"\n{'✓ cùng ngân sách step: ' + str(next(iter(budgets.values()))) if len(set(budgets.values())) == 1 else '⚠ ngân sách KHÁC nhau: ' + str(budgets)}")

## 9. Chấm mọi adapter, bốn nhóm điểm, và **phán quyết**

Fine-tune được chấm với prompt **ngây thơ** (không phải prompt (b)): cái fine-tune mua
cho bạn chính là hành vi đã chuyển vào trọng số nên prompt co lại được — điều này làm
phần cost ở §10 có ý nghĩa.

In [ ]:
def score_adapter(key, version, *, load_in_4bit=False, full=True):
    adir = ADAPTERS / key
    if not (adir / "adapter_model.safetensors").exists():
        print(f"bỏ qua {version}: chưa có {adir}")
        return None
    model, tk = load_model(load_in_4bit=load_in_4bit)
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, str(adir))
    FastLanguageModel.for_inference(model)
    sc = eval_pass(model, tk, version, NAIVE_PROMPT, full=full)
    del model
    gc.collect(); torch.cuda.empty_cache()
    return sc

score_adapter("correct", V_C)
if RUN_REPLAY and (ADAPTERS / "correct_replay" / "adapter_model.safetensors").exists():
    score_adapter("correct_replay", V_CR)

order = [v for v in (V_A, V_B, V_C, V_CR) if v in SCORES]
table = comparison_table({v: SCORES[v] for v in order})
print(md_table(table))

verdicts = {}
for v in (V_C, V_CR):
    if v not in SCORES:
        continue
    verd = regression_gate(SCORES[v], SCORES[V_B])
    verdicts[v] = verd
    print("=" * 60, f"\n{v}: {'PASSED' if verd.passed else 'FAILED'}")
    for r in verd.reasons:
        print("  -", r)

cands = [v for v in (V_C, V_CR) if v in verdicts]
passed = [v for v in cands if verdicts[v].passed]
winner = max(passed, key=lambda v: SCORES[v].target) if passed else \
         (max(cands, key=lambda v: SCORES[v].target + SCORES[v].regression) if cands else None)
print(f"\nbản chọn để so cost/serve: {winner}  (qua cổng: {bool(passed)})")

json.dump({"comparison": table, "winner": winner, "passed_gate": winner in passed,
          "verdicts": {v: verdicts[v].reasons for v in verdicts}, "holdout": HOLD},
         open(WORK / "results" / "verdict.json", "w", encoding="utf-8"),
         ensure_ascii=False, indent=2)

### 9.1 Giải phẫu — ba cấu hình sai, chấm trên thang đo tác vụ (không phải train-loss)

In [ ]:
autopsy = [{"run": k, "version": v, "target": round(SCORES[v].target, 4),
           "format": round(SCORES[v].format, 4), "latency_ms": round(SCORES[v].latency_ms, 1)}
          for k, v in (("correct", V_C), ("correct_replay", V_CR)) if v in SCORES]
for key in ("attn_only", "wrong_lr", "qlora"):
    sc = score_adapter(key, key, load_in_4bit=(key == "qlora"), full=False)
    if sc is None:
        continue
    autopsy.append({"run": key, "version": key, "target": round(sc.target, 4),
                    "format": round(sc.format, 4), "latency_ms": round(sc.latency_ms, 1)})
print(md_table(autopsy))
json.dump(autopsy, open(WORK / "results" / "autopsy.json", "w", encoding="utf-8"),
         ensure_ascii=False, indent=2)

loss_order = [r["run"] for r in sorted((r for r in runs_rows if r.get("final_train_loss") is not None),
                                       key=lambda r: r["final_train_loss"])]
task_order = [r["run"] for r in sorted(autopsy, key=lambda r: -r["target"])]
print(f"\nxếp theo train loss (thấp->cao):  {loss_order}")
print(f"xếp theo điểm target (cao->thấp): {task_order}")
if loss_order and task_order and loss_order[0] != task_order[0]:
    print("→ Hai thứ tự KHÁC nhau — xếp hạng bằng train-loss sẽ chọn sai run.")

### 9.2 Định tính — bắt buộc có ca THUA

In [ ]:
if winner:
    qual = []
    for i, (p, r) in enumerate(zip(PREDS[winner]["target"], target)):
        base_p = PREDS[V_B]["target"][i]
        qual.append({"i": i, "ticket": r["input"][:60],
                    "b": round(triage_field_accuracy(base_p, r["label"]), 2),
                    "ft": round(triage_field_accuracy(p, r["label"]), 2),
                    "ft_pred": p.replace(chr(10), " ")[:80]})
    qual.sort(key=lambda x: (x["ft"] - x["b"], x["ft"]))
    print("--- 3 ca fine-tune THUA/kém nhất so với (b) ---")
    print(md_table(qual[:3], ["i", "ticket", "b", "ft", "ft_pred"]))
    print("\n--- 3 ca fine-tune THẮNG rõ nhất ---")
    print(md_table(qual[-3:], ["i", "ticket", "b", "ft", "ft_pred"]))
    json.dump(qual, open(WORK / "results" / "qualitative.json", "w", encoding="utf-8"),
             ensure_ascii=False, indent=2)

## 10. Latency & cost — nửa còn lại của câu "có nên fine-tune?"

Số ở đây là **phép tính trên số đo** (`mean_latency_ms`, token đếm được ở §6). Giá là
**tham số giả định** — sửa ở §2 rồi chạy lại ô này. Lưu ý: `mean_latency_ms` là
throughput theo batch ở batch size chấm điểm, so được **giữa các phiên bản** nhưng
không phải báo giá production; và self-host giả định GPU luôn có việc.

In [ ]:
def cost_row(version):
    st = STATS[version]
    gpu_sec = st["mean_latency_ms"] / 1000
    self_host_usd_per_1k = gpu_sec * 1000 / 3600 * GPU_HOURLY_USD
    api_usd_per_1k = (st["prompt_tokens_mean"] * API_IN_USD_MTOK +
                      st["new_tokens_mean"] * API_OUT_USD_MTOK) / 1e6 * 1000
    return {"version": version, "latency_ms": st["mean_latency_ms"],
            "prompt_tok": st["prompt_tokens_mean"], "out_tok": st["new_tokens_mean"],
            "self_host_usd_per_1k": round(self_host_usd_per_1k, 4),
            "api_usd_per_1k": round(api_usd_per_1k, 4)}

cost_rows = [cost_row(v) for v in order]
print(md_table(cost_rows))

train_seconds = None
key_of = {V_C: "correct", V_CR: "correct_replay"}
if winner in key_of and key_of[winner] in seen:
    train_seconds = seen[key_of[winner]].get("train_seconds")
train_usd = (train_seconds / 3600 * GPU_HOURLY_USD) if train_seconds else None

if winner and V_B in STATS:
    base_row = cost_row(V_B)
    win_row = cost_row(winner)
    saving_per_1k = base_row["api_usd_per_1k"] - win_row["self_host_usd_per_1k"]
    breakeven_requests = (train_usd / saving_per_1k * 1000) if (train_usd and saving_per_1k > 0) else None
    print(f"\n(b) API cost: ${base_row['api_usd_per_1k']}/1k  vs  {winner} self-host: "
          f"${win_row['self_host_usd_per_1k']}/1k  ->  tiết kiệm ${saving_per_1k:.4f}/1k request")
    if train_usd:
        print(f"chi phí train {winner}: ${train_usd:.3f}")
    if breakeven_requests:
        print(f"break-even: ~{breakeven_requests:.0f} request để hoàn vốn train")
    elif saving_per_1k <= 0:
        print("⚠ self-host KHÔNG rẻ hơn API ở giá giả định hiện tại.")
    if winner not in passed:
        print("⚠ Bản này KHÔNG qua cổng hồi quy ở §9 — số break-even chỉ trả lời "
              "'rẻ hơn khi nào', không trả lời 'có nên ship'.")

all_seconds = sum(r["train_seconds"] for r in runs_rows if r.get("train_seconds"))
print(f"\ntổng thời gian train mọi run: {all_seconds/60:.0f} phút "
      f"= ${all_seconds/3600*GPU_HOURLY_USD:.3f}")
json.dump({"rows": cost_rows, "winner": winner, "train_usd": train_usd,
          "all_runs_train_usd": round(all_seconds/3600*GPU_HOURLY_USD, 4)},
         open(WORK / "results" / "cost.json", "w", encoding="utf-8"),
         ensure_ascii=False, indent=2)

## 11. Chạy thử — cùng câu hỏi, mọi phiên bản, đặt cạnh nhau

Ba ticket (thẳng · hai ý xung đột · ngoài miền) + ba câu tổng quát — chỗ **quên thảm
hoạ** hiện ra bằng mắt nếu bản fine-tune trả lời câu tổng quát bằng JSON triage.

In [ ]:
SAMPLE_TICKETS = [
    ("trong miền", "Mình mua tai nghe bluetooth mã đơn DH998877, hộp còn nguyên nhưng nghe một bên rất rè. Mình muốn đổi cái khác, gấp nhé!"),
    ("hai ý xung đột", "Đơn ND221100 giao chậm 5 ngày rồi mà bàn ủi hơi nước lại bị móp. Shop hoàn tiền cho mình luôn được không?"),
    ("ngoài miền", "Shop có bán cà phê hạt Arabica không, và giao tới Đà Lạt mất bao lâu?"),
]
SAMPLE_GENERAL = ["Thủ đô của Nhật Bản là thành phố nào?",
                  "Giải thích ngắn gọn vì sao trời có mưa.",
                  "Viết một câu cảm ơn khách hàng đã mua hàng."]

sample_results = {v: {} for v in order}
for key, v in (("correct", V_C), ("correct_replay", V_CR)):
    adir = ADAPTERS / key
    if v not in order or not (adir / "adapter_model.safetensors").exists():
        continue
    model, tk = load_model()
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, str(adir))
    FastLanguageModel.for_inference(model)
    tp, _ = generate_measured(model, tk, [t for _, t in SAMPLE_TICKETS], NAIVE_PROMPT,
                              batch_size=len(SAMPLE_TICKETS), warmup=False)
    gp, _ = generate_measured(model, tk, SAMPLE_GENERAL, None, max_new_tokens=96,
                              batch_size=len(SAMPLE_GENERAL), warmup=False)
    sample_results[v] = {"tickets": tp, "general": gp}
    del model
    gc.collect(); torch.cuda.empty_cache()

for i, (kind, ticket) in enumerate(SAMPLE_TICKETS):
    print("=" * 70, f"\nTICKET [{kind}] {ticket}")
    for v in order:
        if sample_results.get(v, {}).get("tickets"):
            print(f"  [{v}] {sample_results[v]['tickets'][i].replace(chr(10),' ')[:200]}")
for i, q in enumerate(SAMPLE_GENERAL):
    print("=" * 70, f"\nCÂU HỎI THƯỜNG NGÀY: {q}")
    for v in order:
        if sample_results.get(v, {}).get("general"):
            print(f"  [{v}] {sample_results[v]['general'][i].replace(chr(10),' ')[:200]}")

json.dump(sample_results, open(WORK / "results" / "samples.json", "w", encoding="utf-8"),
         ensure_ascii=False, indent=2)

## 12. (Tuỳ chọn) Merge adapter thắng vào base

`RUN_MERGE=False` mặc định vì merge ghi thêm vài GB xuống `/kaggle/working`. Có assert
điểm không tụt sau merge — tụt thì đừng deploy.

In [ ]:
if RUN_MERGE and winner in key_of:
    from peft import PeftModel
    model, tk = load_model()
    model = PeftModel.from_pretrained(model, str(ADAPTERS / key_of[winner]))
    FastLanguageModel.for_inference(model)
    n_chk = min(len(target), 20)
    chk = target[:n_chk]
    pre, _ = generate_measured(model, tk, [r["input"] for r in chk], NAIVE_PROMPT, warmup=False)
    before = sum(triage_field_accuracy(p, r["label"]) for p, r in zip(pre, chk)) / n_chk
    merged = model.merge_and_unload()
    post, _ = generate_measured(merged, tk, [r["input"] for r in chk], NAIVE_PROMPT, warmup=False)
    after = sum(triage_field_accuracy(p, r["label"]) for p, r in zip(post, chk)) / n_chk
    TOL = 0.01
    print(f"before={before:.4f}  after={after:.4f}  delta={after-before:+.4f}")
    assert after - before >= -TOL, f"điểm TỤT {before-after:.4f} sau merge (ngưỡng {TOL})"
    merged.save_pretrained(ADAPTERS / "merged")
    tk.save_pretrained(ADAPTERS / "merged")
    del model, merged
    gc.collect(); torch.cuda.empty_cache()
else:
    print("RUN_MERGE=False hoặc chưa có winner hợp lệ — bỏ merge.")

## 13. Cổng kiểm tra trước khi nộp + REPORT.md + zip

In [ ]:
checks = []
def chk(name, ok, detail=""):
    checks.append({"check": name, "pass": bool(ok), "detail": detail})
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f" — {detail}" if detail else ""))

chk("corpus khớp checksum", not integrity["drift"], str(integrity["drift"] or "không"))
chk("không chạy SMOKE mode", not SMOKE, f"eval_limit={EVAL_LIMIT}, 4bit_default={LOAD_IN_4BIT}")
chk("mask: câu trả lời trong loss", mask_proof["answer_is_supervised"])
chk("mask: câu hỏi bị che", mask_proof["question_is_masked"],
    f"supervised_fraction={mask_proof['supervised_fraction']:.3f}")
chk("replay đã tẩy trùng", decon["exact_collisions"] == 0 and decon["worst_jaccard"] <= JACCARD_THRESHOLD,
    f"worst_jaccard={decon['worst_jaccard']}")
budget_vals = {r["run"]: r["max_steps"] for r in runs_rows}
chk("mọi run cùng ngân sách step", len(set(budget_vals.values())) <= 1, str(budget_vals))
chk("có phán quyết cho ít nhất một fine-tune", bool(verdicts),
    ", ".join(f"{v}={'PASS' if verdicts[v].passed else 'FAIL'}" for v in verdicts))
chk("fine-tune thắng (b) và giữ được năng lực chung", bool(winner) and winner in passed)
REQUIRED = ["template_check.json", "mask_proof.json", "token_stats.json",
           "replay_manifest.json", "baselines_frozen.json", "verdict.json",
           "cost.json", "autopsy.json", "samples.json"]
missing = [f for f in REQUIRED if not (WORK / "results" / f).exists()]
chk("đủ artifact bắt buộc", not missing, f"thiếu: {missing or 'không'}")

n_fail = sum(1 for c in checks if not c["pass"])
print(f"\n{len(checks)-n_fail}/{len(checks)} PASS" +
      ("  ✅ sẵn sàng nộp" if not n_fail else f"  ❌ {n_fail} mục cần sửa"))

In [ ]:
lines = [
    "# Day-21 Track-3 — LoRA fine-tune (unsloth, Qwen3.5-4B) cho phân loại ticket",
    "",
    f"- model: `{MODEL_ID}` · r={LORA_R} alpha={LORA_ALPHA} lr={LORA_LR} · "
    f"{STEPS} step, effective batch {PER_DEVICE_BATCH*GRAD_ACCUM} (mọi run giống nhau)",
    f"- mask: assistant-only, {mask_proof['supervised_fraction']:.1%} token vào loss",
    f"- eval: target n={len(target)}, regression n={len(regression)}, holdout n={len(holdout)}"
    + ("  ⚠ SMOKE MODE" if SMOKE else ""),
    "",
    "## 1. Bốn nhóm điểm", "", md_table(table), "",
    f"- (b) − (a) = **{SCORES[V_B].target - SCORES[V_A].target:+.3f}** target "
    "(prompt engineering thuần, không fine-tune)",
]
if winner:
    lines += [f"- {winner} − (b) = **{SCORES[winner].target - SCORES[V_B].target:+.3f}** target, "
              f"**{SCORES[winner].regression - SCORES[V_B].regression:+.3f}** regression",
              f"- cổng hồi quy: **{'PASS' if winner in passed else 'FAIL'}**"]
lines += ["", "## 2. Cấu hình sai (thang đo tác vụ)", "", md_table(autopsy), "",
          f"train-loss order: `{' < '.join(loss_order)}`",
          f"target order:     `{' > '.join(task_order)}`", "",
          "## 3. Cost", "", md_table(cost_rows), "",
          "## 4. Kết luận", "",
          f"- Cổng kiểm tra: {len(checks)-n_fail}/{len(checks)} PASS.",
          "- TODO: bạn sẽ ship bản nào và vì sao? Nếu (b) đủ tốt, câu trả lời đúng "
          "có thể là 'không fine-tune'.", ""]
(WORK / "REPORT.md").write_text("\n".join(lines), encoding="utf-8")
print(f"đã ghi {WORK/'REPORT.md'}  ({len(lines)} dòng)")

In [ ]:
import zipfile
SUB = WORK / "submission"
SUB.mkdir(exist_ok=True)
zpath = SUB / f"lab21_unsloth_qwen35_4b{'_smoke' if SMOKE else ''}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted((WORK / "results").rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(WORK))
    if (WORK / "REPORT.md").exists():
        z.write(WORK / "REPORT.md", "REPORT.md")
    for adir in sorted(ADAPTERS.glob("*")):
        if adir.name == "merged" or not adir.is_dir():
            continue
        for p in sorted(adir.rglob("*")):
            if p.is_file() and p.stat().st_size < 200 * 1024 * 1024:
                z.write(p, p.relative_to(WORK))
print(f"{zpath}  ({zpath.stat().st_size/1024**2:.1f} MB)")
print("✅ PASS" if not n_fail else f"❌ {n_fail} mục FAIL — xem lại trước khi nộp.")